In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.dummy import DummyClassifier

In [5]:
# =============================================================
# 1. ABERTURA E EXAME DOS DADOS
# =============================================================
df = pd.read_csv('/datasets/users_behavior.csv')
 
print("=== Primeiras linhas ===")
print(df.head())
 
print("\n=== Informações gerais ===")
print(df.info())
 
print("\n=== Estatísticas descritivas ===")
print(df.describe())
 
print("\n=== Valores ausentes ===")
print(df.isnull().sum())
 
print("\n=== Distribuição da variável alvo ===")
print(df['is_ultra'].value_counts(normalize=True))

=== Primeiras linhas ===
   calls  minutes  messages   mb_used  is_ultra
0   40.0   311.90      83.0  19915.42         0
1   85.0   516.75      56.0  22696.96         0
2   77.0   467.66      86.0  21060.45         0
3  106.0   745.53      81.0   8437.39         1
4   66.0   418.74       1.0  14502.75         0

=== Informações gerais ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB
None

=== Estatísticas descritivas ===
             calls      minutes     messages       mb_used     is_ultra
count  3214.000000  3214.000000  3214.000000   3214.000000  3214.000000
mean     63.038892   438.208787    38.281269 

In [6]:
# =============================================================
# 2. DIVISÃO DOS DADOS: TREINO / VALIDAÇÃO / TESTE
# Proporção: 60% treino | 20% validação | 20% teste
# =============================================================
features = df.drop(['is_ultra'], axis=1)
target = df['is_ultra']
 
# Primeiro: separa 20% para teste
features_train_val, features_test, target_train_val, target_test = train_test_split(
    features, target, test_size=0.20, random_state=54321
)
 
# Depois: dos 80% restantes, separa 25% para validação (= 20% do total)
features_train, features_valid, target_train, target_valid = train_test_split(
    features_train_val, target_train_val, test_size=0.25, random_state=54321
)
 
print("\n=== Tamanho dos conjuntos ===")
print(f"Treinamento:  {features_train.shape[0]} amostras")
print(f"Validação:    {features_valid.shape[0]} amostras")
print(f"Teste:        {features_test.shape[0]} amostras")


=== Tamanho dos conjuntos ===
Treinamento:  1928 amostras
Validação:    643 amostras
Teste:        643 amostras


In [7]:
# =============================================================
# 3. INVESTIGAÇÃO DE MODELOS E HIPERPARÂMETROS
# =============================================================
 
# --- 3.1 Árvore de Decisão ---
print("\n=== Árvore de Decisão ===")
best_dt_score = 0
best_dt_depth = 0
 
for depth in range(1, 11):
    model_dt = DecisionTreeClassifier(max_depth=depth, random_state=54321)
    model_dt.fit(features_train, target_train)
    score = model_dt.score(features_valid, target_valid)
    print(f"  max_depth={depth} | Acurácia validação: {score:.4f}")
    if score > best_dt_score:
        best_dt_score = score
        best_dt_depth = depth
 
print(f"\n  >> Melhor Árvore de Decisão: max_depth={best_dt_depth} | Acurácia={best_dt_score:.4f}")
 
 
# --- 3.2 Floresta Aleatória ---
print("\n=== Floresta Aleatória ===")
best_rf_score = 0
best_rf_est = 0
best_rf_depth = 0
 
for n_est in range(10, 51, 10):
    for depth in range(1, 11):
        model_rf = RandomForestClassifier(
            n_estimators=n_est, max_depth=depth, random_state=54321
        )
        model_rf.fit(features_train, target_train)
        score = model_rf.score(features_valid, target_valid)
        if score > best_rf_score:
            best_rf_score = score
            best_rf_est = n_est
            best_rf_depth = depth
 
print(f"  >> Melhor Floresta Aleatória: n_estimators={best_rf_est} | max_depth={best_rf_depth} | Acurácia={best_rf_score:.4f}")
 
 
# --- 3.3 Regressão Logística ---
print("\n=== Regressão Logística ===")
model_lr = LogisticRegression(random_state=54321, solver='liblinear', max_iter=1000)
model_lr.fit(features_train, target_train)
lr_score = model_lr.score(features_valid, target_valid)
print(f"  >> Acurácia validação: {lr_score:.4f}")
 
 
# --- Resumo da validação ---
print("\n=== Resumo - Acurácia no Conjunto de Validação ===")
print(f"  Árvore de Decisão:   {best_dt_score:.4f}  (max_depth={best_dt_depth})")
print(f"  Floresta Aleatória:  {best_rf_score:.4f}  (n_estimators={best_rf_est}, max_depth={best_rf_depth})")
print(f"  Regressão Logística: {lr_score:.4f}")


=== Árvore de Decisão ===
  max_depth=1 | Acurácia validação: 0.7760
  max_depth=2 | Acurácia validação: 0.7932
  max_depth=3 | Acurácia validação: 0.8087
  max_depth=4 | Acurácia validação: 0.8040
  max_depth=5 | Acurácia validação: 0.8180
  max_depth=6 | Acurácia validação: 0.8134
  max_depth=7 | Acurácia validação: 0.8149
  max_depth=8 | Acurácia validação: 0.8149
  max_depth=9 | Acurácia validação: 0.8149
  max_depth=10 | Acurácia validação: 0.7978

  >> Melhor Árvore de Decisão: max_depth=5 | Acurácia=0.8180

=== Floresta Aleatória ===
  >> Melhor Floresta Aleatória: n_estimators=30 | max_depth=9 | Acurácia=0.8398

=== Regressão Logística ===
  >> Acurácia validação: 0.7325

=== Resumo - Acurácia no Conjunto de Validação ===
  Árvore de Decisão:   0.8180  (max_depth=5)
  Floresta Aleatória:  0.8398  (n_estimators=30, max_depth=9)
  Regressão Logística: 0.7325


In [14]:
# =============================================================
# 4. AVALIAÇÃO NO CONJUNTO DE TESTE
# =============================================================
features_final = pd.concat([features_train, features_valid])
target_final = pd.concat([target_train, target_valid])

final_rf = RandomForestClassifier(n_estimators=best_rf_est, max_depth=best_rf_depth, random_state=54321)
final_rf.fit(features_final, target_final)
rf_test = final_rf.score(features_test, target_test)

print(f"Acurácia no teste: {rf_test:.4f}")

Acurácia no teste: 0.7854


In [15]:
# =============================================================
# 5. PROVA DE SANIDADE
# =============================================================
print("\n=== Prova de Sanidade - Comparação com Modelo Ingênuo ===")

dummy = DummyClassifier(strategy='most_frequent', random_state=54321)
dummy.fit(features_final, target_final)
dummy_score = dummy.score(features_test, target_test)

print(f"  Acurácia do modelo ingênuo (mais frequente): {dummy_score:.4f}")
print(f"  Acurácia do nosso modelo (Random Forest):    {rf_test:.4f}")
print(f"  Ganho sobre o modelo ingênuo:                +{rf_test - dummy_score:.4f}")
print("\n  ✅ Nosso modelo é significativamente melhor que o modelo ingênuo.")


=== Prova de Sanidade - Comparação com Modelo Ingênuo ===
  Acurácia do modelo ingênuo (mais frequente): 0.6641
  Acurácia do nosso modelo (Random Forest):    0.7854
  Ganho sobre o modelo ingênuo:                +0.1213

  ✅ Nosso modelo é significativamente melhor que o modelo ingênuo.
